# Amazon ECS MCP 서버(Fargate)를 AgentCore Gateway에 연결하기

이 실습에서는 프라이빗 VPC 내부의 Amazon ECS Fargate에 [FastMCP](https://github.com/jlowin/fastmcp) 서버를 배포한 다음, 내부 ALB를 통한 managed VPC egress를 사용하여 [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)에 연결합니다.

MCP 서버에는 VPC와 연결된 Route 53 프라이빗 호스팅 영역의 **프라이빗 도메인**을 통해 접근할 수 있습니다. VPC에서 **Private DNS**를 활성화하면(기본값), AgentCore Gateway의 managed Resource Gateway가 VPC의 DNS 해석기를 통해 도메인을 확인합니다.

VPC egress, 인증서 요구 사항 및 Private DNS에 관한 배경 정보는 [프로젝트 README](../README.md), [Managed VPC Resource README 문서](../01-managed-vpc-resource/README.md), [사전 요구 사항](../00-prerequisites/)을 참조하세요.

![아키텍처](./images/ecs-fargate.png)


## 사전 요구 사항

- [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb) 완료(VPC + AgentCore Gateway 배포)
- Docker 실행 중(CDK 컨테이너 이미지 빌드용)
- ALB에서 TLS 종료에 사용할 [ACM 퍼블릭 인증서](../00-prerequisites/create-acm-public-certificate.md)


## 1단계: 종속성 설치 및 라이브러리 가져오기


In [ ]:
import os
from pathlib import Path

# 프로젝트 루트로 이동
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

!pip install --force-reinstall -q -r requirements.txt

In [ ]:
import json
import os
import time

import boto3
from utils.utils import get_token

# 실습 0에서 변수 복원
%store -r ACCOUNT_A_ID
%store -r ACCOUNT_A_PROFILE
%store -r GATEWAY_ID
%store -r GATEWAY_URL
%store -r USER_POOL_ID
%store -r USER_POOL_CLIENT_ID
%store -r TOKEN_ENDPOINT_URL
%store -r OAUTH_SCOPES
%store -r VPC_USW2_ID
%store -r VPC_USW2_PRIVATE_SUBNETS

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID

REGION = "us-west-2"
session = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION)
agentcore = session.client("bedrock-agentcore-control")

# Cognito 클라이언트 암호 가져오기
cognito = session.client("cognito-idp")
client_desc = cognito.describe_user_pool_client(UserPoolId=USER_POOL_ID, ClientId=USER_POOL_CLIENT_ID)
CLIENT_SECRET = client_desc["UserPoolClient"]["ClientSecret"]

print(f"Account:    {ACCOUNT_A_ID}")
print(f"Region:     {REGION}")
print(f"Gateway ID: {GATEWAY_ID}")
print(f"VPC ID:     {VPC_USW2_ID}")

In [ ]:
CERT_ARN = input("ACM public certificate ARN: ").strip()
DOMAIN = input("Domain name covered by the certificate (e.g., api.internal.yourcompany.com): ").strip()

assert CERT_ARN.startswith("arn:aws:acm:"), "Invalid certificate ARN"
assert not DOMAIN.startswith("http"), "Domain should not include http:// or https://"
assert "." in DOMAIN, "Domain must contain at least one dot"
assert " " not in DOMAIN, "Domain must not contain whitespace"

print(f"Cert ARN: {CERT_ARN}")
print(f"Domain:   {DOMAIN}")

## 2단계: ECS Fargate에 MCP 서버 배포

이 CDK 스택은 다음 리소스를 배포합니다.
- 포트 8000에서 FastMCP(echo, add, get_time 도구)를 실행하고 Cloud Map(`mcp.local`)에 등록되는 **ECS Fargate 서비스**
- ACM 퍼블릭 인증서를 사용하여 TLS를 종료하는 HTTPS 리스너(포트 443)가 있는 **내부 ALB**
- VPC와 연결되고 ALB를 가리키는 apex Alias 레코드가 포함된 `<DOMAIN>`이라는 이름의 **Route 53 프라이빗 호스팅 영역**
- 테스트를 위한 SSM Session Manager 지원 **배스천 인스턴스**(t3.micro)

VPC 내부에서 `<DOMAIN>`은 Private DNS를 통해 ALB의 프라이빗 IP로 확인됩니다. AgentCore Gateway의 Resource Gateway는 이 확인 경로를 사용합니다.


In [ ]:
!cdk deploy McpEcs \
    -c "publicCertArn={CERT_ARN}" \
    -c "privateDomain={DOMAIN}" \
    --profile {ACCOUNT_A_PROFILE} \
    --require-approval never \
    --outputs-file ecs-outputs.json

In [ ]:
with open("ecs-outputs.json") as f:
    ecs_outputs = json.load(f)["McpEcs"]

ALB_DNS = ecs_outputs["AlbDnsName"]
ALB_SG_ID = ecs_outputs["AlbSgId"]
PRIVATE_DOMAIN = ecs_outputs["PrivateDomain"]

print(f"Private FQDN: {PRIVATE_DOMAIN}  (resolves via Private DNS inside VPC → ALB)")
print(f"ALB DNS:      {ALB_DNS}")
print(f"ALB SG:       {ALB_SG_ID}")

## 3단계: AgentCore Gateway 대상 생성

[Managed VPC Resource 실습](../01-managed-vpc-resource/)을 사용하여 Gateway 대상을 생성합니다. 대상이 프라이빗 FQDN을 가리키도록 설정하면 Private DNS가 이를 VPC 내부의 ALB로 확인합니다.

- **대상 URL** (`https://{DOMAIN}/mcp`) - 프라이빗 호스팅 영역을 통해 ALB의 프라이빗 IP로 확인되며 ALB의 퍼블릭 인증서와 일치합니다.
- **`managedVpcResource`** - Resource Gateway ENI가 포트 443에서 ALB에 도달할 수 있도록 VPC, 서브넷 및 ALB의 보안 그룹을 지정합니다.

> **보안 그룹:** Resource Gateway ENI가 포트 443에서 ALB에 도달할 수 있도록 ALB의 보안 그룹을 `securityGroupIds`에 전달합니다.


In [ ]:
TARGET_ENDPOINT = f"https://{DOMAIN}/mcp"

print(f"Target endpoint: {TARGET_ENDPOINT}  (resolves via Private DNS inside the VPC)")

response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="ecs-mcp-server",
    description="MCP server on ECS Fargate via internal ALB and managed VPC egress (Private DNS)",
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": TARGET_ENDPOINT,
            }
        }
    },
    privateEndpoint={
        "managedVpcResource": {
            "vpcIdentifier": VPC_USW2_ID,
            "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
            "endpointIpAddressType": "IPV4",
            "securityGroupIds": [ALB_SG_ID],
        }
    },
)

TARGET_ID = response["targetId"]
print(f"\nTarget ID: {TARGET_ID}")
print(f"Status:    {response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nTarget is active!")
        print(f"  Managed resources: {target.get('privateEndpoint', {})}")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

## 4단계: AgentCore Gateway를 통해 MCP 서버 호출

Cognito에서 액세스 토큰을 가져온 다음 Gateway를 통해 MCP 서버의 도구를 호출합니다.


In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]
print(f"Access token obtained (expires in {token_response['expires_in']}s)")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

# 사용 가능한 도구 나열
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
print("Available tools:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# 프라이빗 MCP 서버에서 "add" 도구 호출
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "ecs-mcp-server___add",
            "arguments": {"a": 5, "b": 3},
        },
        "id": 2,
    },
)
print("Result of add(5, 3):")
print(json.dumps(response.json(), indent=2))

## 5단계(선택 사항): 주가 MCP 서버 연결

CDK 스택은 **경로 기반 라우팅**을 사용하여 동일한 ALB에 두 번째 MCP 서버인 **모의 주가 서버**를 배포했습니다.
- `/mcp` → 기존 MCP 서버(echo, add, get_time)
- `/stock-mcp/*` → 주가 MCP 서버(get_stock_price, get_market_summary)

두 서버는 동일한 ALB, 인증서, 프라이빗 호스팅 영역 및 보안 그룹을 공유합니다. VPC/서브넷/SG 구성이 일치하므로 두 번째 대상은 VPC의 동일한 Resource Gateway를 재사용합니다.

In [ ]:
STOCK_TARGET_ENDPOINT = f"https://{DOMAIN}/stock-mcp/"

print(f"Stock target endpoint: {STOCK_TARGET_ENDPOINT}")

stock_response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="ecs-stock-mcp",
    description="Stock price MCP server on ECS Fargate via same ALB (path-based routing, Private DNS)",
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": STOCK_TARGET_ENDPOINT,
            }
        }
    },
    privateEndpoint={
        "managedVpcResource": {
            "vpcIdentifier": VPC_USW2_ID,
            "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
            "endpointIpAddressType": "IPV4",
            "securityGroupIds": [ALB_SG_ID],
        }
    },
)

STOCK_TARGET_ID = stock_response["targetId"]
print(f"\nStock Target ID: {STOCK_TARGET_ID}")
print(f"Status:          {stock_response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=STOCK_TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nStock target is active!")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

# 주가 MCP 도구 나열
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
tools = response.json().get("result", {}).get("tools", [])
stock_tools = [t for t in tools if t["name"].startswith("ecs-stock-mcp___")]
print(f"Stock MCP tools ({len(stock_tools)}):")
for t in stock_tools:
    print(f"  {t['name']}: {t.get('description', '')[:80]}")

# 주가 가져오기
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "ecs-stock-mcp___get_stock_price",
            "arguments": {"symbol": "AAPL"},
        },
        "id": 2,
    },
)
print("\nAAPL stock price:")
print(json.dumps(response.json(), indent=2))

## 정리

1. Gateway 대상 삭제
2. CDK 스택 삭제(ALB SG는 유지됨)
3. 유지된 ALB 보안 그룹 삭제

> **참고:** AgentCore의 managed Resource Gateway ENI가 ALB 보안 그룹을 계속 참조할 수 있으므로 스택 삭제 중에도 해당 보안 그룹은 유지됩니다. Gateway 대상이 완전히 제거된 후 수동으로 삭제하세요.


In [ ]:
# # 1단계: Gateway target 삭제
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# print(f"Deleting target: {TARGET_ID}")
# while True:
#     try:
#         t = agentcore.get_gateway_target(
#             gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID
#         )
#         print(f"  Status: {t['status']}")
#         time.sleep(15)
#     except agentcore.exceptions.ResourceNotFoundException:
#         print("  Target deleted.")
#         break

# # stock target 삭제(5단계에서 생성한 경우)
# try:
#     agentcore.delete_gateway_target(
#         gatewayIdentifier=GATEWAY_ID, targetId=STOCK_TARGET_ID
#     )
#     print(f"Deleting stock target: {STOCK_TARGET_ID}")
#     while True:
#         try:
#             t = agentcore.get_gateway_target(
#                 gatewayIdentifier=GATEWAY_ID, targetId=STOCK_TARGET_ID
#             )
#             print(f"  Status: {t['status']}")
#             time.sleep(15)
#         except agentcore.exceptions.ResourceNotFoundException:
#             print("  Stock target deleted.")
#             break
# except NameError:
#     pass  # Stock target was not created (Step 5 skipped)

In [ ]:
# # 2단계: CDK stack 제거(ALB SG는 유지되며 다음 cell에서 삭제)
# !ACCOUNT_A_ID={ACCOUNT_A_ID} cdk destroy McpEcs \
#     -c "publicCertArn={CERT_ARN}" \
#     -c "privateDomain={DOMAIN}" \
#     --profile {ACCOUNT_A_PROFILE} --force

In [ ]:
# # 3단계: 유지된 ALB security group 삭제
# ec2_client = session.client("ec2")
# try:
#     ec2_client.delete_security_group(GroupId=ALB_SG_ID)
#     print(f"Deleted ALB security group: {ALB_SG_ID}")
# except ec2_client.exceptions.ClientError as e:
#     if "DependencyViolation" in str(e):
#         print(f"SG {ALB_SG_ID} still has dependencies. Wait a few minutes and retry.")
#     else:
#         raise